# Overreaction as an Indicator for Momentum in Algorithmic Trading: A Case of AAPL Stocks

This notebook implements the strategy described in the paper "Overreaction as an indicator for momentum in algorithmic trading: A Case of AAPL stocks" by Szymon Lis, Robert Ślepaczuk, and Paweł Sakowski. The strategy uses high-frequency emotional information and machine learning methods to predict and monetize short-term market overreactions as momentum signals.

**Paper Citation:**
Lis, S., Ślepaczuk, R., & Sakowski, P. (2026). Overreaction as an indicator for momentum in algorithmic trading: A Case of AAPL stocks. arXiv preprint arXiv:2602.18912.

**Abstract:**
This paper investigates whether short-term market overreactions can be systematically predicted and monetized as momentum signals using high-frequency emotional information and modern machine learning methods. Focusing on Apple Inc. (AAPL), we construct a comprehensive intraday dataset that combines volatility normalized returns with transformer-based emotion features extracted from Twitter messages. Overreactions are defined as extreme return realizations relative to contemporaneous volatility and transaction costs and are modeled as a three-class prediction problem. We evaluate the performance of several nonlinear classifiers, including XGBoost, Random Forests, Deep Neural Networks, and Bidirectional LSTMs, across multiple intraday frequencies (1, 5, 10, and 15 minute data). Model outputs are translated into trading strategies and assessed using risk-adjusted performance measures and formal statistical tests. The results show that machine learning models significantly outperform benchmark overreaction rules at ultra short horizons, while classical behavioral momentum effects dominate at intermediate frequencies, particularly around 10 minutes. Explainability analysis based on SHAP reveals that volatility and negative emotions, especially fear and sadness, play a central role in driving predicted overreactions. Overall, the findings demonstrate that emotion-driven overreactions contain a predictable structure that can be exploited by machine learning models, offering new insights into the behavioral origins of intraday momentum and the interaction between sentiment, volatility, and algorithmic trading.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In this phase, we define the configuration parameters for our trading strategy. This includes the ticker universe, parameters for the strategy, and a hypothesis comment block.

In [ ]:
# Configuration
UNIVERSE = ['AAPL']
START_DATE = '2020-01-01'
END_DATE = '2023-01-01'
HYPOTHESIS = 'Short-term market overreactions can be systematically predicted and monetized as momentum signals using high-frequency emotional information and modern machine learning methods.'

## Phase 2 — Data Download & Feature Computation

In this phase, we download market data using yfinance, compute the necessary factors and features, and perform cross-sectional normalization.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start=START_DATE, end=END_DATE, interval='1m')

# Compute features
data['returns'] = data['Close'].pct_change()
data['volatility'] = data['returns'].rolling(window=21).std()
data['overreaction'] = np.where(data['returns'] > 2 * data['volatility'], 1, np.where(data['returns'] < -2 * data['volatility'], -1, 0))

# Cross-sectional normalization
data['overreaction'] = data['overreaction'] / data['overreaction'].abs().sum()

## Phase 3 — Signal Generation, Position Sizing, & Portfolio Construction

In this phase, we generate trading signals based on the overreaction feature, determine position sizes, and construct the portfolio.

In [ ]:
# Signal generation
data['signal'] = data['overreaction'].shift(1)

# Position sizing
data['position'] = np.where(data['signal'] == 1, 1, np.where(data['signal'] == -1, -1, 0))

# Portfolio construction
data['portfolio_return'] = data['returns'] * data['position']

## Phase 4 — Vectorized Backtest

In this phase, we perform a vectorized backtest of the strategy, ensuring no look-ahead bias by shifting signals forward by 1 period.

In [ ]:
# Vectorized backtest
data['cumulative_return'] = (1 + data['portfolio_return']).cumprod()
data['cumulative_return'] = data['cumulative_return'].fillna(1)

## Phase 5 — Performance Metrics

In this phase, we calculate performance metrics such as Sharpe ratio, Sortino ratio, Calmar ratio, maximum drawdown, and plot the equity curve.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Performance metrics
annual_return = data['portfolio_return'].mean() * 252
annual_volatility = data['portfolio_return'].std() * np.sqrt(252)
sharpe_ratio = annual_return / annual_volatility
sortino_ratio = annual_return / data['portfolio_return'][data['portfolio_return'] < 0].std() * np.sqrt(252)
max_drawdown = (data['cumulative_return'].cummax() - data['cumulative_return']) / data['cumulative_return'].cummax().shift(1)
calmar_ratio = annual_return / max_drawdown.max()

# Equity curve plot
plt.plot(data['cumulative_return'])
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.show()

# Print performance metrics
print(f'Annual Return: {annual_return:.2%}')
print(f'Annual Volatility: {annual_volatility:.2%}')
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown.max():.2%}')

## Phase 6 — Monitoring Stub

In this phase, we create a function that prints daily P&L and current positions given live data.

In [ ]:
# Monitoring stub
def monitor_positions(data):
    current_position = data['position'].iloc[-1]
    daily_pnl = data['portfolio_return'].iloc[-1]
    print(f'Current Position: {current_position}')
    print(f'Daily P&L: {daily_pnl:.2%}')

# Example usage
monitor_positions(data)